# Smoke Test Notebook

This notebook verifies the full pipeline end to end on a single MVTec AD category before running full experiments.

**Goal:** Confirm that:
1. Environment and dependencies are correctly installed
2. Anomalib loads and runs Dinomaly correctly
3. Model outputs can be extracted and fed into our evaluation pipeline
4. `evaluation/metrics.py` computes correct values on real model output

**Dataset:** MVTec AD (downloads automatically via Anomalib)  
**Model:** Dinomaly with DINOv2-Base/14  
**Category:** bottle (single category for speed)

In [ ]:
# Check GPU and environment
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

# Clone repo into Drive so it persists between sessions
repo_path = '/content/drive/MyDrive/BachelorsThesis'
if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
else:
    # Pull latest changes if repo already exists
    !git -C {repo_path} pull

# Add repo to Python path so we can import our modules
sys.path.insert(0, repo_path)
print(f"Repo path added to sys.path: {repo_path}")

In [ ]:
# Install dependencies
!pip install anomalib einops timm kornia lightning scikit-image -q

# Verify Anomalib version
import anomalib
print(f"Anomalib version: {anomalib.__version__}")

## Step 1: Run Dinomaly on MVTec AD (bottle category)

Anomalib handles data downloading, preprocessing, training and inference automatically.
We use the Engine API which is Anomalib's high level interface.

In [ ]:
from anomalib.models import Dinomaly
from anomalib.data import MVTecAD
from anomalib.engine import Engine

# Configure dataset — bottle category, downloads automatically to /content/mvtec
datamodule = MVTecAD(
    root='/content/mvtec',
    category='bottle',
    train_batch_size=8,
    eval_batch_size=8,
    num_workers=2,
)

# Configure model — uses DINOv2-Base/14 by default
model = Dinomaly()

# Configure engine — handles training and evaluation
engine = Engine(
    max_epochs=1,          # just 1 epoch for smoke test
    accelerator='gpu',
    devices=1,
)


print(f"Model: {model.__class__.__name__}")
print(f"Dataset: MVTec AD - bottle")
print("Ready to train")

In [ ]:
# Train and evaluate
engine.fit(model=model, datamodule=datamodule)
engine.test(model=model, datamodule=datamodule)

print("Training and evaluation complete")

## Step 2: Extract Scores and Run Our Metrics

Extract anomaly scores from Anomalib's output and feed into our evaluation pipeline.

In [ ]:
import pandas as pd
import numpy as np
from evaluation.metrics import compute_i_auroc, compute_all_metrics, MVTEC_CONFIG

# Extract predictions from the engine
predictions = engine.predict(model=model, datamodule=datamodule)

# Build standardised dataframe from Anomalib output
rows = []
for batch in predictions:
    for i in range(len(batch.image_path)):
        rows.append({
            'image_path':  batch.image_path[i],
            'image_score': batch.pred_score[i].item(),
            'label':       int(batch.gt_label[i].item()),
            'has_mask':    batch.gt_mask is not None,
            'category':    'bottle',
        })

df = pd.DataFrame(rows)
print(f"Total images: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nSample rows:")
print(df[['image_path', 'image_score', 'label']].head())

In [ ]:
# Compute metrics using our pipeline
metrics = compute_all_metrics(df, config=MVTEC_CONFIG, compute_pixel=False)

print("Results:")
for k, v in metrics.items():
    print(f"  {k}: {round(v, 4)}")
    
# Clear GPU memory after model run
torch.cuda.empty_cache()
import gc
gc.collect()
print("\nGPU memory cleared")

## Smoke Test Complete

If all cells ran without errors and I-AUROC printed a sensible value (above 0.5), 
the pipeline is working correctly end to end.

Next step: run full experiments with all three models on Real-IAD.

## AnomalyDINO Smoke Test

Testing AnomalyDINO pipeline — training-free nearest-neighbour method.
No training loop required, just feature extraction from normal images.
Same MVTec AD bottle category for direct comparison.

In [ ]:
# Clear GPU memory from Dinomaly run first
torch.cuda.empty_cache()
import gc
gc.collect()
print("GPU memory cleared")

# Import AnomalyDINO
from anomalib.models import AnomalyDINO

# Use same datamodule as before — bottle category
# No need to redefine datamodule, it is already loaded

# Configure AnomalyDINO model
model_dino = AnomalyDINO()

# Configure engine — AnomalyDINO is training-free
# but we still use engine.fit() to build the memory bank
engine_dino = Engine(
    max_epochs=1,
    accelerator='gpu',
    devices=1,
)

print(f"Model: {model_dino.__class__.__name__}")
print("Ready — AnomalyDINO is training-free, fit() just builds memory bank")